<a href="https://colab.research.google.com/github/spbromberg/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [2]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** order line item

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [3]:
import pandas as pd

# Turn the nested lines list into one row per line item
# Copy the order number and vendor ID onto each line-item row
lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)

# Check the required number of rows and total quantity
print('Rows:', len(lines))
print('Total quantity:', lines['qty'].sum())

lines

Rows: 4
Total quantity: 7


,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [4]:
# Convert the vendor list into a table and join it to the line-item table.
# A left join keeps every order line, even if its vendor is missing.
# indicator=True creates _merge to show whether each vendor matched.
joined = lines.merge(
    pd.DataFrame(vendors_json),
    on='vendor_id',
    how='left',
    indicator=True
)

# Keep only rows whose vendor ID did not appear in vendors_json.
unmatched = joined[joined['_merge'] == 'left_only']

# Report the missing vendor and the total quantity connected to it.
print('Unmatched vendor:', unmatched['vendor_id'].iloc[0])
print('Units attached to unmatched vendor:', unmatched['qty'].sum())

joined

Unmatched vendor: V-07
Units attached to unmatched vendor: 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [5]:
# Replace the missing vendor name with a clear label.
# This keeps the unmatched vendor visible in the groupby result.
joined['vendor_name'] = joined['name'].fillna('Unknown vendor')

# Add quantities for each vendor name and sort from highest to lowest.
vendor_totals = (
    joined.groupby('vendor_name')['qty']
    .sum()
    .sort_values(ascending=False)
)

vendor_totals

,qty
vendor_name,
Hoos Burgers,3
Unknown vendor,3
Cav Merch,1


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [6]:
# Add quantities for each zone and sort from highest to lowest.
# Rows with no zone are not included in this result.
zone_totals = (
    joined.groupby('zone')['qty']
    .sum()
    .sort_values(ascending=False)
)

# Print the zone that sold the most items and explain the missing units.
print('Zone with the most items:', zone_totals.index[0])
print('Items sold in that zone:', zone_totals.iloc[0])
print('Total quantity included in zone totals:', zone_totals.sum())
print('Difference:', lines['qty'].sum() - zone_totals.sum(),
      'units from V-07, which has no matched zone.')

zone_totals

Zone with the most items: A
Items sold in that zone: 4
Total quantity included in zone totals: 4
Difference: 3 units from V-07, which has no matched zone.


,qty
zone,
A,4


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [7]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# Keeping only orders that have a lines key
# This prevents json_normalize from crashing on the order with no lines
orders_with_lines = [order for order in ragged_json if 'lines' in order]

# Flatten the available line items into one row per item
# A missing qty stays as NaN, which is different from a real quantity of 0
ragged_lines = pd.json_normalize(
    orders_with_lines,
    record_path='lines',
    meta=['order', 'vendor_id']
)

ragged_lines

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Fries,NaN,6,V-01


### Q6 — Validate

In [8]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**a)** “One row per line item” is useful because each item and its quantity becomes readily available for analysis. This shape makes questions such as “How many items did each vendor sell?” or “Which zone sold the most items?” easy to answer with groupby. With one row per order, the items would remain inside a nested list, making totals by item, vendor, or zone more difficult to calculate.

**b)** A missing quantity means the API did not provide a quantity, while a quantity of zero means the item was recorded and its actual quantity was zero. They should not be treated the same. If missing quantities are replaced with zeros, totals may look complete even though information is missing from the dataset. This could incorrectly lower sales totals and hide a data-quality issues.